## 0. Entorno

In [1]:
import pandas as pd
import openpyxl
import numpy as np
import seaborn as sns
import matplotlib as plt
import plotly.express as px
import plotly.io as pio
from pathlib import Path
import json
import requests
import time
import mysql.connector

#Sirve para cargar el .env con la credenciales de la AEMET
from dotenv import load_dotenv
import os

In [40]:
import os
os.getcwd()

'/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-'

## 1. Creación de tablas CSV de clima a partir de API AEMET

In [42]:
#carga CSV 'NASA'
clima_NASA = pd.read_csv(
    '/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/POWER_Point_Monthly_20050101_20241231_040d42N_003d70W_LST.csv',
    skiprows=13,
    na_values=['-999', '-999.0']
)

In [60]:
#carga de fichero .xlsx 'ESYRCE'
agriC_ESP = pd.read_excel(r'/Users/rosamariasierraalmeria/Documents/GitHub/Impacto-del-clima-en-la-agricultura-/data/Dinaweb2024-2.xlsx')

In [37]:
#Carga de datos de la API de la AEMET
#Carga de la API Key

load_dotenv('apikey.env')
API_KEY = os.getenv("AEMET_API_KEY")

#Creación de diccionario que relaciona cada estación de la AEMET con el cultivo que me interesa
estaciones = {
    "cereales": "2422",
    "vinedo": "9145Y",
    "olivar": "5298X",
    "citricos": "8416"
}

In [3]:
#Función para descargar json
def descargar_json_aemet(
    api_key,
    estacion,
    fecha_ini,
    fecha_fin,
    fichero_salida
):  
    endpoint = (
        f"https://opendata.aemet.es/opendata/api/"
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{fecha_ini}T00:00:00UTC/"
        f"fechafin/{fecha_fin}T23:59:59UTC/"
        f"estacion/{estacion}"
    )

    headers = {
        "api_key": api_key
    }

    respuesta = requests.get(
        endpoint,
        headers=headers
    )

    metadata = respuesta.json()
   
    print(metadata)

    if metadata["estado"] != 200:
        print(metadata)
        return

    datos_url = metadata["datos"]

    datos = requests.get(datos_url).json()

    respuesta_datos = requests.get(datos_url)

    if respuesta_datos.status_code != 200:
     print(print(
        f"Fallo en {estacion} "
        f"{fecha_ini} - {fecha_fin}"
        ))
     return

    try:
        datos = respuesta_datos.json()

    except Exception:
        print(
        f"No se pudo leer JSON: "
        f"{fecha_ini} - {fecha_fin}"
     )
        return

    with open(
        fichero_salida,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            datos,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(
        f"Guardado {fichero_salida}"
    )

In [ ]:
for anio in range(2004, 2024):

    PAUSA = 20

    descargar_json_aemet(
        API_KEY,
        "2422",
        f"{anio}-01-01",
        f"{anio}-06-30",
        f"data/AEMET/valladolid/valladolid_{anio}_S1.json"
    )

    time.sleep(PAUSA)

    descargar_json_aemet(
        API_KEY,
        "2422",
        f"{anio}-07-01",
        f"{anio}-12-31",
        f"data/AEMET7valladolid/valladolid_{anio}_S2.json"
    )

    time.sleep(PAUSA)

print("Descarga finalizada")

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/8774e1c7', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2004_S1.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/ad36dfe4', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2004_S2.json
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/851a5907', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid_2005_S2.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/7381d001', 'metadatos': 'https:/

In [137]:
#Descarga solo de un semestre
descargar_json_aemet(
    API_KEY,
    "2422",
    "2024-01-01",
    "2024-06-30",
    "data/AEMET/valladolid/valladolid_2024_S1.json"
)

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/c4ee5f40', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valladolid/valladolid_2024_S1.json


In [36]:
#Bucle para comprobar si las estaciones de la AEMET recogen datos desde las fechas que necesito
for anio in range(2004, 2025):

    endpoint = (
        f"https://opendata.aemet.es/opendata/api/"
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{anio}-01-01T00:00:00UTC/"
        f"fechafin/{anio}-06-30T23:59:59UTC/"
        f"estacion/8416"
    )

    respuesta = requests.get(
        endpoint,
        headers={"api_key": API_KEY}
    )

    metadata = respuesta.json()

    if metadata["estado"] == 200:
        print(f"✅ {anio}")
    else:
        print(f"❌ {anio}")

✅ 2004
✅ 2005
✅ 2006
✅ 2007
✅ 2008
✅ 2009
✅ 2010
✅ 2011
✅ 2012
✅ 2013
✅ 2014
✅ 2015
✅ 2016
✅ 2017
✅ 2018
✅ 2019
✅ 2020
✅ 2021
✅ 2022
✅ 2023
✅ 2024


In [139]:
#Unión de todos los .JSON para crear la tabla de clima de Valladolid

ruta = Path("data/AEMET/valladolid")
dfs = []

for archivo in ruta.glob("*.json"):

    df = pd.read_json(archivo)
    dfs.append(df)

clima_valladolid = pd.concat(
    dfs,
    ignore_index=True
)

clima_valladolid.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2010-01-01,2422,VALLADOLID,VALLADOLID,734,"5,0","0,7","1,8",23:59,"8,2",...,"2,7","930,3",24,"918,3",00,83.0,90.0,Varias,68.0,15:15
1,2010-01-02,2422,VALLADOLID,VALLADOLID,734,"4,6",Ip,"1,2",01:00,"8,0",...,"1,0","933,5",11,"930,1",01,88.0,98.0,04:30,77.0,23:59
2,2010-01-03,2422,VALLADOLID,VALLADOLID,734,"6,7","8,1","5,2",08:15,"8,2",...,"0,0","931,0",00,"927,5",15,93.0,100.0,21:00,77.0,Varias
3,2010-01-04,2422,VALLADOLID,VALLADOLID,734,"6,6","9,3","5,2",23:59,"8,0",...,"0,0","927,8",00,"916,3",24,95.0,100.0,Varias,92.0,Varias
4,2010-01-05,2422,VALLADOLID,VALLADOLID,734,"4,3","0,9","2,0",23:59,"6,6",...,"1,1","919,3",24,"915,8",02,81.0,100.0,Varias,71.0,14:15


In [140]:
#Limpieza de datos
clima_valladolid["fecha"] = pd.to_datetime(
    clima_valladolid["fecha"]
)

for col in ["tmed", "tmin", "tmax"]:

    clima_valladolid[col] = (
        clima_valladolid[col]
        .str.replace(",", ".")
        .astype(float)
    )

clima_valladolid["prec"] = (
    clima_valladolid["prec"]
    .str.replace(",", ".")
)

In [150]:
aemet_valladolid_anual = (
    clima_valladolid
    .groupby("anio")
    .agg(
        temp_media=("tmed", "mean"),
        temp_min=("tmin", "mean"),
        temp_max=("tmax", "mean"),
        precipitacion=("prec", "sum")
    )
    .reset_index()
)

In [ ]:
aemet_valladolid_anual.to_csv(
    "data/AEMET/tablas_limpias/aemet_valladolid.csv",
    index=False
)

In [ ]:
for anio in range(2004, 2024):

    PAUSA = 20

    descargar_json_aemet(
        API_KEY,
        "9145Y",
        f"{anio}-01-01",
        f"{anio}-06-30",
        f"data/AEMET/logroño/logroño_{anio}_S1.json"
    )

    time.sleep(PAUSA)

    descargar_json_aemet(
        API_KEY,
        "9145Y",
        f"{anio}-07-01",
        f"{anio}-12-31",
        f"data/AEMET/logroño/logroño_{anio}_S2.json"
    )

    time.sleep(PAUSA)

print("Descarga finalizada")

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/09579be9', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/logroño/logroño_2004_S1.json
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/705c5ba1', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/logroño/logroño_2005_S1.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/27893a22', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/logroño/logroño_2005_S2.json
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente

In [169]:
descargar_json_aemet(
    API_KEY,
    "9145Y",
    "2021-01-01",
    "2021-06-30",
    "data/AEMET/logroño/logroño_2021_S1.json"
)

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/a1adf954', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/logroño/logroño_2021_S1.json


In [170]:
#Unión de todos los .JSON para crear la tabla de clima de Logroño
ruta = Path("data/AEMET/logroño")
dfs = []

for archivo in ruta.glob("*.json"):

    df = pd.read_json(archivo)
    dfs.append(df)

clima_logroño = pd.concat(
    dfs,
    ignore_index=True
)

clima_logroño.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,dir,velmedia,racha,horaracha,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2020-01-01,9145Y,CENICERO,LA RIOJA,429,"-0,2","0,0","-2,6",05:52,"2,1",13:31,10.0,"1,1","3,9",15:20,98.0,100.0,Varias,93.0,15:10
1,2020-01-02,9145Y,CENICERO,LA RIOJA,429,"-1,0","0,0","-1,7",08:33,"-0,2",15:24,11.0,"1,4","4,4",21:40,100.0,100.0,Varias,99.0,Varias
2,2020-01-03,9145Y,CENICERO,LA RIOJA,429,"3,5","0,0","-1,4",Varias,"8,4",15:36,24.0,"2,5","8,1",12:30,88.0,100.0,Varias,79.0,Varias
3,2020-01-04,9145Y,CENICERO,LA RIOJA,429,"7,7","0,0","4,9",03:03,"10,5",13:56,28.0,"2,8","7,2",13:20,85.0,97.0,03:30,71.0,14:00
4,2020-01-05,9145Y,CENICERO,LA RIOJA,429,"7,0","0,0","0,9",23:54,"13,1",15:36,26.0,"3,3","7,2",03:50,70.0,88.0,08:40,44.0,15:40


In [171]:
#Limpieza de datos
clima_logroño["fecha"] = pd.to_datetime(
    clima_logroño["fecha"]
)

for col in ["tmed", "tmin", "tmax"]:

    clima_logroño[col] = (
        clima_logroño[col]
        .str.replace(",", ".")
        .astype(float)
    )

clima_logroño["prec"] = (
    clima_logroño["prec"]
    .str.replace(",", ".")
)

In [174]:
clima_logroño["fecha"] = pd.to_datetime(
    clima_logroño["fecha"]
)

clima_logroño["anio"] = (
    clima_logroño["fecha"]
    .dt.year
)

In [175]:
aemet_logroño_anual = (
    clima_logroño
    .groupby("anio")
    .agg(
        temp_media=("tmed", "mean"),
        temp_min=("tmin", "mean"),
        temp_max=("tmax", "mean"),
        precipitacion=("prec", "sum")
    )
    .reset_index()
)

In [ ]:
aemet_logroño_anual.to_csv(
    "data/AEMET/tablas_limpias/aemet_logroño.csv",
    index=False
)

In [7]:
for anio in range(2004, 2024):

    PAUSA = 20

    descargar_json_aemet(
        API_KEY,
        "5298X",
        f"{anio}-01-01",
        f"{anio}-06-30",
        f"data/AEMET/jaen/jaen_{anio}_S1.json"
    )

    time.sleep(PAUSA)

    descargar_json_aemet(
        API_KEY,
        "5298X",
        f"{anio}-07-01",
        f"{anio}-12-31",
        f"data/AEMET/jaen/jaen_{anio}_S2.json"
    )

    time.sleep(PAUSA)

print("Descarga finalizada")

{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/677f21f8', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/jaen/jaen_2004_S2.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/41c425d9', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/jaen/jaen_2005_S1.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/b47ccc87', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/jaen/jaen_2005_S2.json
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/a6dc81c5', 'metadatos': 'https://op

In [27]:
descargar_json_aemet(
    API_KEY,
    "5298X",
    "2009-07-01",
    "2009-12-31",
    "data/AEMET/jaen/jaen_2009_S2.json"
)

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/bcd8aa77', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/jaen/jaen_2009_S2.json


In [ ]:
#Unión de todos los .JSON para crear la tabla de clima de Jaén
ruta = Path("data/AEMET/jaen")
dfs = []

for archivo in ruta.glob("*.json"):

    df = pd.read_json(archivo)
    dfs.append(df)

clima_jaen = pd.concat(
    dfs,
    ignore_index=True
)

clima_jaen.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,horaracha,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2023-07-01,5298X,ANDÚJAR,JAEN,202,"28,8","0,0","19,7",04:40,"38,0",...,23:50,"993,0",08,"988,1",Varias,31.0,67.0,04:40,18.0,Varias
1,2023-07-02,5298X,ANDÚJAR,JAEN,202,"31,0","0,0","22,2",05:30,"39,7",...,17:40,"992,7",Varias,"988,5",17,37.0,74.0,05:20,19.0,Varias
2,2023-07-03,5298X,ANDÚJAR,JAEN,202,"31,4","0,0","23,4",05:00,"39,5",...,21:20,"993,2",08,"989,3",Varias,32.0,58.0,04:50,17.0,15:30
3,2023-07-04,5298X,ANDÚJAR,JAEN,202,"30,4","0,0","21,6",05:50,"39,1",...,12:20,"993,6",08,"989,7",18,31.0,69.0,05:50,11.0,16:10
4,2023-07-05,5298X,ANDÚJAR,JAEN,202,"28,1","0,0","17,0",05:30,"39,2",...,16:30,"992,9",Varias,"987,8",18,23.0,54.0,Varias,12.0,Varias


In [29]:
#Limpieza de datos
clima_jaen["fecha"] = pd.to_datetime(
    clima_jaen["fecha"]
)

for col in ["tmed", "tmin", "tmax"]:

    clima_jaen[col] = (
        clima_jaen[col]
        .str.replace(",", ".")
        .astype(float)
    )

clima_jaen["prec"] = (
    clima_jaen["prec"]
    .str.replace(",", ".")
)

In [30]:
clima_jaen["fecha"] = pd.to_datetime(
    clima_jaen["fecha"]
)

clima_jaen["anio"] = (
    clima_jaen["fecha"]
    .dt.year
)

In [31]:
aemet_jaen_anual = (
    clima_jaen
    .groupby("anio")
    .agg(
        temp_media=("tmed", "mean"),
        temp_min=("tmin", "mean"),
        temp_max=("tmax", "mean"),
        precipitacion=("prec", "sum")
    )
    .reset_index()
)

In [32]:
aemet_jaen_anual.to_csv(
    "data/AEMET/tablas_limpias/aemet_jaen.csv",
    index=False
)

In [38]:
for anio in range(2004, 2024):

    PAUSA = 20

    descargar_json_aemet(
        API_KEY,
        "8416",
        f"{anio}-01-01",
        f"{anio}-06-30",
        f"data/AEMET/valencia/valencia_{anio}_S1.json"
    )

    time.sleep(PAUSA)

    descargar_json_aemet(
        API_KEY,
        "8416",
        f"{anio}-07-01",
        f"{anio}-12-31",
        f"data/AEMET/valencia/valencia_{anio}_S2.json"
    )

    time.sleep(PAUSA)

print("Descarga finalizada")

{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/84fe8f44', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valencia/valencia_2004_S2.json
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{'descripcion': 'L�mite de peticiones o caudal por minuto excedido para este usuario. Espere al siguiente minuto.', 'estado': 429}
{

In [66]:
descargar_json_aemet(
    API_KEY,
    "8416",
    "2023-01-01",
    "2023-06-30",
    "data/AEMET/valencia/valencia_2023_S1.json"
)

{'descripcion': 'exito', 'estado': 200, 'datos': 'https://opendata.aemet.es/opendata/sh/0d0025f6', 'metadatos': 'https://opendata.aemet.es/opendata/sh/b3aa9d28'}
Guardado data/AEMET/valencia/valencia_2023_S1.json


In [67]:
#Unión de todos los .JSON para crear la tabla de clima de Valencia
ruta = Path("data/AEMET/valencia")
dfs = []

for archivo in ruta.glob("*.json"):

    df = pd.read_json(archivo)
    dfs.append(df)

clima_valencia = pd.concat(
    dfs,
    ignore_index=True
)

clima_valencia.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,sol,presMax,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin
0,2008-07-01,8416,VALÈNCIA,VALENCIA,13,"26,1","0,0","21,2",05:00,"31,0",...,"11,4","1015,7",0,"1012,0",18,62.0,NaN,NaN,NaN,NaN
1,2008-07-02,8416,VALÈNCIA,VALENCIA,13,"25,4","0,0","21,1",04:50,"29,8",...,"8,8","1010,8",0,"1005,7",19,78.0,NaN,NaN,NaN,NaN
2,2008-07-03,8416,VALÈNCIA,VALENCIA,13,"26,6","0,0","22,9",05:30,"30,2",...,"9,8","1014,2",24,"1005,8",03,71.0,NaN,NaN,NaN,NaN
3,2008-07-04,8416,VALÈNCIA,VALENCIA,13,"25,8","0,0","22,9",00:10,"28,7",...,"11,4","1017,0",9,"1013,5",18,70.0,NaN,NaN,NaN,NaN
4,2008-07-05,8416,VALÈNCIA,VALENCIA,13,"26,0","0,0","22,3",04:30,"29,6",...,"11,0","1013,8",0,"1008,5",19,78.0,NaN,NaN,NaN,NaN


In [68]:
#Limpieza de datos
clima_valencia["fecha"] = pd.to_datetime(
    clima_valencia["fecha"]
)

for col in ["tmed", "tmin", "tmax"]:

    clima_valencia[col] = (
        clima_valencia[col]
        .str.replace(",", ".")
        .astype(float)
    )

clima_valencia["prec"] = (
    clima_valencia["prec"]
    .str.replace(",", ".")
)

In [69]:
clima_valencia["fecha"] = pd.to_datetime(
    clima_valencia["fecha"]
)

clima_valencia["anio"] = (
    clima_valencia["fecha"]
    .dt.year
)

In [70]:
aemet_valencia_anual = (
    clima_valencia
    .groupby("anio")
    .agg(
        temp_media=("tmed", "mean"),
        temp_min=("tmin", "mean"),
        temp_max=("tmax", "mean"),
        precipitacion=("prec", "sum")
    )
    .reset_index()
)

In [72]:
aemet_valencia_anual.to_csv(
    "data/AEMET/tablas_limpias/aemet_valencia.csv",
    index=False
)